In [ ]:
import os


import torch
import numpy as np
import evaluate
from dataclasses import dataclass
from typing import Any, Dict, List
from datasets import load_dataset, Audio, ClassLabel
from transformers import (
    AutoFeatureExtractor,
    AutoModelForAudioClassification,
    AutoConfig,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import classification_report


MODEL_ID = "facebook/mms-300m"
DATASET_ID = "badrex/nnti-dataset-full"
AUDIO_COLUMN = "audio_filepath"      
LABEL_COLUMN = "language"      
MAX_DURATION_SECONDS = 10.0 


print("Loading dataset...")
full_dataset = load_dataset(DATASET_ID)

# Cast to 16kHz
full_dataset["train"] = full_dataset["train"].cast_column(AUDIO_COLUMN, Audio(sampling_rate=16000))
full_dataset["validation"] = full_dataset["validation"].cast_column(AUDIO_COLUMN, Audio(sampling_rate=16000))

# unique labels and cast to ClassLabel 
unique_labels = sorted(full_dataset["train"].unique(LABEL_COLUMN))
full_dataset["train"] = full_dataset["train"].cast_column(LABEL_COLUMN, ClassLabel(names=unique_labels))
full_dataset["validation"] = full_dataset["validation"].cast_column(LABEL_COLUMN, ClassLabel(names=unique_labels))

val_test_split = full_dataset["validation"].train_test_split(test_size=0.5, seed=42)

dataset = {
    "train": full_dataset["train"],
    "validation": val_test_split["train"],
    "test": val_test_split["test"]
}

print(f"Dataset loaded with existing splits:")
print(f"Train samples: {len(dataset['train'])}")
print(f"Validation samples: {len(dataset['validation'])}")
print(f"Test samples: {len(dataset['test'])}")

labels = dataset["train"].features[LABEL_COLUMN].names
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}
num_labels = len(labels)


print("Initializing feature extractor...")
feature_extractor = AutoFeatureExtractor.from_pretrained(
    MODEL_ID,
    do_normalize=True,
    return_attention_mask=True,
)
target_sampling_rate = feature_extractor.sampling_rate
max_length = int(target_sampling_rate * MAX_DURATION_SECONDS)

def preprocess_function(examples):
    audio_arrays = [x["array"] for x in examples[AUDIO_COLUMN]]
    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=target_sampling_rate,
        max_length=max_length,
        truncation=True,
        return_attention_mask=True,
    )
    inputs["labels"] = examples[LABEL_COLUMN]
    # Store length for group_by_length
    inputs["length"] = [len(x) for x in inputs["input_values"]]
    return inputs

print("Preprocessing datasets...")
encoded_train = dataset["train"].map(preprocess_function, remove_columns=dataset["train"].column_names, batched=True)
encoded_val = dataset["validation"].map(preprocess_function, remove_columns=dataset["validation"].column_names, batched=True)
encoded_test = dataset["test"].map(preprocess_function, remove_columns=dataset["test"].column_names, batched=True)

print("Label example:", encoded_train[0]["labels"])
print("Max label:", max(encoded_train["labels"]))
print("Num labels:", num_labels)


@dataclass
class DataCollatorForAudioClassification:
    feature_extractor: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        input_features = [
            {
                "input_values": feature["input_values"],
                "attention_mask": feature["attention_mask"],
            }
            for feature in features
        ]
        label_features = [feature["labels"] for feature in features]

        batch = self.feature_extractor.pad(
            input_features,
            padding=True,
            return_tensors="pt",
        )
        batch["labels"] = torch.tensor(label_features, dtype=torch.int64)
        return batch

data_collator = DataCollatorForAudioClassification(feature_extractor=feature_extractor)


print("Initializing model...")
config = AutoConfig.from_pretrained(MODEL_ID)
config.num_labels = num_labels
config.label2id = label2id
config.id2label = id2label

Loading dataset...


README.md:   0%|          | 0.00/679 [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/382M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/383M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/311M [00:00<?, ?B/s]

...

Map:   0%|          | 0/8689 [00:00<?, ? examples/s]

Map:   0%|          | 0/1650 [00:00<?, ? examples/s]

Map:   0%|          | 0/1650 [00:00<?, ? examples/s]

Label example: 8
Max label: 21
Num labels: 22
Initializing model...


config.json: 0.00B [00:00, ?B/s]

In [ ]:
model = AutoModelForAudioClassification.from_pretrained(
    MODEL_ID,
    config=config,
    ignore_mismatched_sizes=True,
)
model.freeze_feature_encoder()


import torch.nn as nn
if hasattr(model, 'projector'):
    nn.init.normal_(model.projector.weight, mean=0.0, std=0.02)
    if model.projector.bias is not None:
        nn.init.zeros_(model.projector.bias)
if hasattr(model, 'classifier'):
    nn.init.normal_(model.classifier.weight, mean=0.0, std=0.02)
    if model.classifier.bias is not None:
        nn.init.zeros_(model.classifier.bias)
print(f"Classifier head reinitialized (std=0.02). Expected initial loss: ~{np.log(num_labels):.2f}")

import evaluate, numpy as np
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=preds, references=labels)

class CorrectedTrainer(Trainer):
    def log(self, logs, *args, **kwargs):
        if "loss" in logs and self.args.gradient_accumulation_steps > 1:
            logs["loss"] = logs["loss"] / self.args.gradient_accumulation_steps
        super().log(logs, *args, **kwargs)


training_args = TrainingArguments(
    output_dir="./mms-300m-nnti-finetuned",
    eval_strategy="epoch",       
    save_strategy="epoch",            
    learning_rate=3e-5,
    per_device_train_batch_size=16,        
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1,        
    num_train_epochs=5,
    warmup_steps=100,           
    logging_steps=10,
    
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss", 
    greater_is_better=False,          
    
    fp16=False,                 
    weight_decay=0.01,
)

trainer = CorrectedTrainer(
    model=model,
    args=training_args,
    train_dataset=encoded_train,
    eval_dataset=encoded_val,         
    processing_class=feature_extractor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
for batch in trainer.get_train_dataloader():
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(**batch)
    print("Initial loss:", outputs.loss.item())
    break


print("Starting training...")
trainer.train()

print("Training complete! Best model (based on validation loss) has been automatically loaded.")


print("\n" + "="*50)
print("RUNNING FINAL EVALUATION ON UNSEEN TEST SET")
print("="*50)

predictions_output = trainer.predict(encoded_test)

logits = predictions_output.predictions
true_labels = predictions_output.label_ids
predicted_labels = np.argmax(logits, axis=-1)

report = classification_report(
    true_labels, 
    predicted_labels, 
    target_names=labels, 
    digits=4             
)

print("\nFinal Test Set Metrics Report:")
print(report)

trainer.save_model("./mms-300m-nnti-final-best")
print("\nFinal Best Model saved to ./mms-300m-nnti-final-best")

pytorch_model.bin:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2ForSequenceClassification LOAD REPORT from: facebook/mms-300m
Key                          | Status     | 
-----------------------------+------------+-
quantizer.weight_proj.bias   | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
projector.bias               | MISSING    | 
classifier.bias              | MISSING    | 
classifier.weight            | MISSING    | 
projector.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Classifier head reinitialized (std=0.02). Expected initial loss: ~3.09


model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

...

Training complete! Best model (based on validation loss) has been automatically loaded.

RUNNING FINAL EVALUATION ON UNSEEN TEST SET



Final Test Set Metrics Report:
              precision    recall  f1-score   support

    assamese     0.4719    0.5060    0.4884        83
     bengali     0.2500    0.2078    0.2270        77
        bodo     0.2933    0.5946    0.3929        74
       dogri     0.1495    0.2424    0.1850        66
    gujarati     0.1698    0.2250    0.1935        80
       hindi     0.2857    0.0588    0.0976        68
     kannada     0.2264    0.1739    0.1967        69
    kashmiri     0.3409    0.1786    0.2344        84
     konkani     0.0000    0.0000    0.0000        79
    maithili     0.4000    0.0253    0.0476        79
   malayalam     0.3604    0.6061    0.4520        66
    manipuri     0.2834    0.6625    0.3970        80
     marathi     0.2114    0.3421    0.2613        76
      nepali     0.0833    0.0323    0.0465        62
        odia     0.3750    0.0423    0.0759        71
     punjabi     0.2500    0.2785    0.2635        79
    sanskrit     0.3913    0.2169    0.2791      

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Final Best Model saved to ./mms-300m-nnti-final-best


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report


print("\n" + "="*50)
print("VALIDATION SET ERROR ANALYSIS")
print("="*50)

val_predictions_output = trainer.predict(encoded_val)

val_logits = val_predictions_output.predictions
val_true_labels = val_predictions_output.label_ids
val_predicted_labels = np.argmax(val_logits, axis=-1)

true_langs = [labels[i] for i in val_true_labels]
pred_langs = [labels[i] for i in val_predicted_labels]


print("\nValidation Classification Report:")
print(
    classification_report(
        val_true_labels,
        val_predicted_labels,
        target_names=labels,
        digits=4,
        zero_division=0
    )
)


cm = confusion_matrix(val_true_labels, val_predicted_labels)

cm_df = pd.DataFrame(cm, index=labels, columns=labels)
print("\nConfusion Matrix (rows=true, cols=predicted):")
print(cm_df)

cm_df.to_csv("validation_confusion_matrix.csv", index=True)


errors_df = pd.DataFrame({
    "true_label_id": val_true_labels,
    "pred_label_id": val_predicted_labels,
    "true_language": true_langs,
    "predicted_language": pred_langs,
})

errors_df = errors_df[errors_df["true_label_id"] != errors_df["pred_label_id"]].copy()

print(f"\nTotal validation samples: {len(val_true_labels)}")
print(f"Total incorrect predictions: {len(errors_df)}")
print(f"Validation accuracy: {(val_true_labels == val_predicted_labels).mean():.4f}")

errors_df.to_csv("validation_errors_only.csv", index=False)

mistake_summary = (
    errors_df
    .groupby(["true_language", "predicted_language"])
    .size()
    .reset_index(name="count")
    .sort_values(["true_language", "count"], ascending=[True, False])
)

print("\nMistake summary: for each TRUE language, which WRONG language was predicted:")
print(mistake_summary)

mistake_summary.to_csv("validation_mistake_summary.csv", index=False)


total_per_true_lang = pd.Series(true_langs).value_counts().rename_axis("true_language").reset_index(name="total_samples")

mistake_summary = mistake_summary.merge(total_per_true_lang, on="true_language", how="left")
mistake_summary["percent_of_that_language"] = 100 * mistake_summary["count"] / mistake_summary["total_samples"]

mistake_summary = mistake_summary.sort_values(
    ["true_language", "percent_of_that_language"],
    ascending=[True, False]
)

print("\nPer-language mistake percentages:")
print(mistake_summary)

mistake_summary.to_csv("validation_mistake_summary_with_percent.csv", index=False)

top_confusions = mistake_summary.sort_values("count", ascending=False).head(20)

print("\nTop 20 most common confusions on validation set:")
print(top_confusions[["true_language", "predicted_language", "count", "percent_of_that_language"]])


VALIDATION SET ERROR ANALYSIS



Validation Classification Report:
              precision    recall  f1-score   support

    assamese     0.5616    0.6119    0.5857        67
     bengali     0.4000    0.2466    0.3051        73
        bodo     0.2968    0.6053    0.3983        76
       dogri     0.2000    0.2619    0.2268        84
    gujarati     0.1389    0.2857    0.1869        70
       hindi     0.4000    0.0732    0.1237        82
     kannada     0.1944    0.0864    0.1197        81
    kashmiri     0.3947    0.2273    0.2885        66
     konkani     0.0000    0.0000    0.0000        71
    maithili     0.5000    0.0563    0.1013        71
   malayalam     0.3269    0.6071    0.4250        84
    manipuri     0.2386    0.6714    0.3521        70
     marathi     0.2323    0.3108    0.2659        74
      nepali     0.2000    0.0682    0.1017        88
        odia     0.5000    0.1266    0.2020        79
     punjabi     0.2105    0.2817    0.2410        71
    sanskrit     0.3095    0.1940    0.2385   